# Lab 4 — Part 0: Build Isaac Lab Container

Build the Isaac Lab training container and push to ECR.
**Run once before `Lab4_RL_Training.ipynb`.** Rebuild only when container files change.

---

## Prerequisites (read before running)

| Requirement | Details |
|-------------|---------|
| **NGC API key** | Get from https://ngc.nvidia.com/setup/api-key — needed to pull the 16 GB base image |
| **GPU quota** | `ml.g5.xlarge` for training jobs (G-family only — P-family lacks RT Cores) |
| **IAM: Secrets Manager** | SageMaker role needs `secretsmanager:GetSecretValue` on `ngc-api-key` (Section 1a fixes this) |
| **IAM: CodeBuild** | SageMaker role needs `codebuild:StartBuild`, `codebuild:BatchGetBuilds`, etc. |
| **IAM: ECR** | SageMaker role needs `ecr:CreateRepository`, `ecr:DescribeRepositories` |
| **Admin role** | `iam:PutRolePolicy` (Section 1a) requires an admin-level role, not the SageMaker role itself |

## External dependencies

- `nvcr.io/nvidia/isaac-lab:2.1.0` — NVIDIA NGC registry (~16 GB, pulled by CodeBuild)
- AWS CodeBuild `standard:7.0` image — used as the build host

## Validated status

> Build has been run end-to-end and succeeded (11 min, image 15.8 GB in ECR).
> The container runs Isaac Lab built-in tasks (`Isaac-Velocity-Flat-Anymal-D-v0`,
> `Isaac-Reach-UR10-v0`) on SageMaker. Custom UR3 tasks are not yet wired into
> the container — this is planned future work.

## Why CodeBuild (not local Docker)

The NGC base image is ~16 GB and is x86-only — it cannot be built on Apple Silicon.
CodeBuild runs on a large cloud instance with the NGC API key from Secrets Manager.


## 0 — Setup

In [ ]:
import boto3, io, json, time, zipfile
from pathlib import Path

# Resolve repo root by walking up from cwd
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != 'aws-physical-ai-toolchain' and REPO_ROOT != REPO_ROOT.parent:
 REPO_ROOT = REPO_ROOT.parent

REGION = boto3.session.Session().region_name or 'us-west-2'
ACCOUNT = boto3.client('sts', region_name=REGION).get_caller_identity()['Account']
BUCKET = f'sagemaker-{REGION}-{ACCOUNT}'
ROLE_ARN = f'arn:aws:iam::{ACCOUNT}:role/service-role/AmazonSageMaker-ExecutionRole-20260607T114524'
ROLE_NAME = ROLE_ARN.split('/')[-1]
ECR_REPO = 'physical-ai/isaac-lab'
ECR_URI = f'{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}'
CB_PROJECT = 'physical-ai-isaac-lab-build'
NGC_SECRET_NAME = 'ngc-api-key'
S3_KEY = 'codebuild/isaac-lab-context.zip'

ecr = boto3.client('ecr', region_name=REGION)
cb = boto3.client('codebuild', region_name=REGION)
sm = boto3.client('secretsmanager', region_name=REGION)
s3 = boto3.client('s3', region_name=REGION)
iam = boto3.client('iam', region_name=REGION)

print(f'Region: {REGION}')
print(f'Account: {ACCOUNT}')
print(f'Bucket: {BUCKET}')
print(f'ECR URI: {ECR_URI}')
print(f'CB project: {CB_PROJECT}')
print(f'Repo root: {REPO_ROOT}')


## 1 — Verify Prerequisites

Checks CodeBuild access, ECR repo existence, and NGC secret availability.
If anything is missing, the relevant Section 1a/1b cells will fix it.

In [ ]:
issues = []

# CodeBuild
try:
 cb.list_projects()
 print(' CodeBuild access OK')
except Exception as e:
 issues.append(f'CodeBuild: {e}')
 print(f' CodeBuild: {e}')

# ECR
try:
 ecr.describe_repositories(repositoryNames=[ECR_REPO])
 print(f" ECR repo '{ECR_REPO}' exists")
except ecr.exceptions.RepositoryNotFoundException:
 print(f" ECR repo '{ECR_REPO}' not found — will be created in Section 2")
except Exception as e:
 issues.append(f'ECR: {e}')
 print(f' ECR: {e}')

# NGC secret
try:
 sm.describe_secret(SecretId=NGC_SECRET_NAME)
 print(f" NGC secret '{NGC_SECRET_NAME}' exists in Secrets Manager")
except sm.exceptions.ResourceNotFoundException:
 print(f" NGC secret '{NGC_SECRET_NAME}' not found — run Section 1b to create it")
except Exception as e:
 print(f" Could not check NGC secret: {e}")

# Container source files in repo
for fpath in [
 REPO_ROOT / 'containers' / 'isaac-lab' / 'Dockerfile',
 REPO_ROOT / 'containers' / 'isaac-lab' / 'sm-train-entrypoint.sh',
 REPO_ROOT / 'containers' / 'isaac-lab' / 'train_entrypoint.py',
]:
 if fpath.exists():
 print(f' {fpath.relative_to(REPO_ROOT)} ({fpath.stat().st_size // 1024} KB)')
 else:
 issues.append(f'Missing: {fpath}')
 print(f' Missing: {fpath}')

print()
if issues:
 print('Fix the issues above before proceeding.')
else:
 print(' All source files present — proceed through Sections 1a→1b→1c→2→3→4→5.')


## 1a — Fix IAM: Attach Secrets Manager Policy to SageMaker Role

CodeBuild's PRE_BUILD phase reads the NGC key from Secrets Manager using the SageMaker
execution role. Without `secretsmanager:GetSecretValue` on that role, CodeBuild fails
with **exit code 254** during the NGC login step.

This cell attaches an inline policy granting exactly the needed permissions.
It is **idempotent** — safe to re-run.

> **Requires admin-level role.** `iam:PutRolePolicy` is not available on the SageMaker
> execution role itself. If this cell fails with `AccessDenied`, attach the policy
> manually in the IAM console:
> IAM → Roles → `AmazonSageMaker-ExecutionRole-20260607T114524` → Add inline policy
> (see `README.md` → IAM Requirements for the exact JSON).

In [ ]:
POLICY_NAME = 'NgcSecretReadForCodeBuild'
SECRET_ARN_PREFIX = f'arn:aws:secretsmanager:{REGION}:{ACCOUNT}:secret:{NGC_SECRET_NAME}'

policy_doc = {
 'Version': '2012-10-17',
 'Statement': [{
 'Sid': 'AllowCodeBuildToReadNgcKey',
 'Effect': 'Allow',
 'Action': [
 'secretsmanager:GetSecretValue',
 'secretsmanager:DescribeSecret',
 ],
 # Wildcard suffix covers the 6-char random suffix Secrets Manager appends
 # (e.g. ngc-api-key-AbCdEf)
 'Resource': f'{SECRET_ARN_PREFIX}*',
 }]
}

try:
 iam.put_role_policy(
 RoleName=ROLE_NAME,
 PolicyName=POLICY_NAME,
 PolicyDocument=json.dumps(policy_doc),
 )
 print(f" Policy '{POLICY_NAME}' attached to '{ROLE_NAME}'")
 print(f" Action: secretsmanager:GetSecretValue + DescribeSecret")
 print(f" Resource: {SECRET_ARN_PREFIX}*")
except iam.exceptions.NoSuchEntityException:
 print(f' Role not found: {ROLE_NAME}')
 print(' Check ROLE_ARN in the Setup cell.')
except Exception as e:
 if 'AccessDenied' in str(e) or 'not authorized' in str(e).lower():
 print(f' Access denied: {e}')
 print()
 print('You need an admin-level role to call iam:PutRolePolicy.')
 print('Attach the policy manually:')
 print(' IAM → Roles → AmazonSageMaker-ExecutionRole-20260607T114524')
 print(' → Add inline policy → JSON → paste from README.md § IAM Requirements')
 else:
 print(f' Unexpected error: {e}')


## 1b — Create NGC API Key Secret

If the `ngc-api-key` secret doesn't exist yet, this cell creates it.
If it already exists, it just confirms and skips.

**Get your NGC API key:** https://ngc.nvidia.com/setup/api-key

Set `NGC_API_KEY` to your key string before running. Leave as `None` to skip
(safe if the secret already exists from a prior run).

In [ ]:
# ------------------------------------------------------------------ #
# Set your NGC API key here before running. #
# Leave as None to skip creation if the secret already exists. #
# ------------------------------------------------------------------ #
NGC_API_KEY = None # e.g. 'YOUR_NGC_API_KEY'

try:
 secret = sm.describe_secret(SecretId=NGC_SECRET_NAME)
 print(f" Secret '{NGC_SECRET_NAME}' already exists — no action needed.")
 print(f" ARN: {secret['ARN']}")
except sm.exceptions.ResourceNotFoundException:
 if NGC_API_KEY is None:
 print(f" Secret '{NGC_SECRET_NAME}' does not exist.")
 print(" Set NGC_API_KEY above and re-run this cell.")
 print(" Get your key at: https://ngc.nvidia.com/setup/api-key")
 else:
 resp = sm.create_secret(
 Name=NGC_SECRET_NAME,
 SecretString=NGC_API_KEY,
 Description='NVIDIA NGC API key for pulling Isaac Lab base image in CodeBuild',
 )
 print(f" Secret created: {resp['ARN']}")
except Exception as e:
 print(f' Unexpected error: {e}')


## 1c — Verify Prerequisites Are Met

Re-run this cell to confirm the IAM policy and NGC secret are in place.
Both must show before proceeding to Section 2.

In [ ]:
POLICY_NAME = 'NgcSecretReadForCodeBuild'
ready = True

# IAM policy check
try:
 iam.get_role_policy(RoleName=ROLE_NAME, PolicyName=POLICY_NAME)
 print(f" IAM policy '{POLICY_NAME}' attached to '{ROLE_NAME}'")
except iam.exceptions.NoSuchEntityException:
 print(f" IAM policy '{POLICY_NAME}' not attached — re-run Section 1a")
 ready = False
except Exception as e:
 # May not be able to read IAM policies from SageMaker role — warn but continue
 print(f" Could not verify IAM policy (may be OK if attached manually): {e}")

# NGC secret readability
try:
 sm.get_secret_value(SecretId=NGC_SECRET_NAME)
 print(f" Secret '{NGC_SECRET_NAME}' is readable")
except sm.exceptions.ResourceNotFoundException:
 print(f" Secret '{NGC_SECRET_NAME}' not found — re-run Section 1b")
 ready = False
except Exception as e:
 print(f" Secret read failed: {e}")
 ready = False

print()
if ready:
 print(' Prerequisites met — proceed to Section 2.')
else:
 print(' Fix the issues above before continuing.')


## 2 — Create ECR Repository

In [ ]:
try:
 repo = ecr.describe_repositories(repositoryNames=[ECR_REPO])['repositories'][0]
 print(f" Already exists: {repo['repositoryUri']}")
except ecr.exceptions.RepositoryNotFoundException:
 repo = ecr.create_repository(repositoryName=ECR_REPO)['repository']
 print(f" Created: {repo['repositoryUri']}")
except Exception as e:
 print(f' ECR error: {e}')


## 3 — Package & Upload Build Context

Creates a zip that CodeBuild uses as the Docker build context.

**Zip structure (matches Dockerfile COPY paths exactly):**
```
buildspec.yml
Dockerfile (from containers/isaac-lab/)
containers/isaac-lab/sm-train-entrypoint.sh
containers/isaac-lab/batch-train-entrypoint.sh
containers/isaac-lab/train_entrypoint.py
training/ (all .py/.yaml, excludes training/data/)
workflows/
```

Key lessons baked into this cell:
- `training/data/` excluded — 4000+ files would bloat the build context
- `BytesIO` is inspected **before** `upload_fileobj` (upload reads to EOF)
- `docker tag ... $ECR_REPO_URI` — no extra `:latest` (URI already has it)
- All files under `containers/isaac-lab/` included so Dockerfile COPYs don't fail

In [ ]:
BUILDSPEC = f'''version: 0.2
phases:
 pre_build:
 commands:
 - echo Logging in to ECR and NGC...
 - REGISTRY=$(echo $ECR_REPO_URI | cut -d/ -f1)
 - aws ecr get-login-password --region $AWS_DEFAULT_REGION | docker login --username AWS --password-stdin $REGISTRY
 - NGC_KEY=$(aws secretsmanager get-secret-value --secret-id {NGC_SECRET_NAME} --region $AWS_DEFAULT_REGION --query SecretString --output text)
 - echo $NGC_KEY | docker login --username '\$oauthtoken' --password-stdin nvcr.io
 build:
 commands:
 - echo Building isaac-lab image...
 - docker build --platform linux/amd64 -t isaac-lab-training .
 - docker tag isaac-lab-training $ECR_REPO_URI
 post_build:
 commands:
 - docker push $ECR_REPO_URI
 - echo Done. Image pushed to $ECR_REPO_URI
'''

buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
 zf.writestr('buildspec.yml', BUILDSPEC)

 # --- Container source files from containers/isaac-lab/ ---
 container_src = REPO_ROOT / 'containers' / 'isaac-lab'
 for fname in ['Dockerfile', 'sm-train-entrypoint.sh',
 'batch-train-entrypoint.sh', 'train_entrypoint.py']:
 fpath = container_src / fname
 if fpath.exists():
 # Dockerfile goes at zip root; entrypoints keep containers/isaac-lab/ prefix
 arc = fname if fname == 'Dockerfile' else f'containers/isaac-lab/{fname}'
 zf.write(fpath, arc)
 print(f' Added: {arc} ({fpath.stat().st_size // 1024} KB)')
 else:
 print(f' Not found (optional): containers/isaac-lab/{fname}')

 # --- training/ directory: all .py and .yaml, exclude training/data/ ---
 training_dir = REPO_ROOT / 'training'
 added = 0
 for p in sorted(training_dir.rglob('*')):
 if not p.is_file():
 continue
 rel = str(p.relative_to(REPO_ROOT))
 if '__pycache__' in rel or p.suffix == '.pyc':
 continue
 if rel.startswith('training/data/'):
 continue # skip — 4000+ teleop data files would bloat the build context
 zf.write(p, rel)
 added += 1
 print(f' Added: training/ ({added} files, data/ excluded)')

 # --- workflows/ directory ---
 workflows_dir = REPO_ROOT / 'workflows'
 wf_added = 0
 if workflows_dir.exists():
 for wf in sorted(workflows_dir.rglob('*')):
 if wf.is_file():
 zf.write(wf, str(wf.relative_to(REPO_ROOT)))
 wf_added += 1
 print(f' Added: workflows/ ({wf_added} files)')

# Inspect zip contents BEFORE upload (upload_fileobj reads to EOF)
buf.seek(0)
with zipfile.ZipFile(buf) as zf:
 names = sorted(zf.namelist())
print(f'\n {len(names)} files total. Key file check:')
must_have = [
 'Dockerfile',
 'containers/isaac-lab/sm-train-entrypoint.sh',
 'training/scripts/launch_rl.py',
]
for f in must_have:
 print(f" {'' if f in names else ' MISSING'} {f}")

# Upload
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, S3_KEY)
print(f'\n Uploaded: s3://{BUCKET}/{S3_KEY}')


## 4 — Create / Update CodeBuild Project

In [ ]:
project_cfg = dict(
 source={
 'type': 'S3',
 'location': f'{BUCKET}/{S3_KEY}',
 'buildspec': 'buildspec.yml',
 },
 artifacts={'type': 'NO_ARTIFACTS'},
 environment={
 'type': 'LINUX_CONTAINER',
 'image': 'aws/codebuild/standard:7.0',
 'computeType': 'BUILD_GENERAL1_2XLARGE', # 128 GB RAM — NGC base is ~16 GB
 'privilegedMode': True,
 'environmentVariables': [
 {'name': 'ECR_REPO_URI', 'value': f'{ECR_URI}:latest'},
 {'name': 'AWS_DEFAULT_REGION', 'value': REGION},
 ],
 },
 serviceRole=ROLE_ARN,
 timeoutInMinutes=90, # NGC pull + layer install can take 60-90 min
)

try:
 cb.create_project(name=CB_PROJECT, **project_cfg)
 print(f' Project created: {CB_PROJECT}')
except cb.exceptions.ResourceAlreadyExistsException:
 # update_project doesn't accept timeoutInMinutes in the same call
 update_cfg = {k: v for k, v in project_cfg.items() if k != 'timeoutInMinutes'}
 cb.update_project(name=CB_PROJECT, **update_cfg)
 print(f' Project updated: {CB_PROJECT}')
except Exception as e:
 print(f' Error: {e}')


## 5 — Start Build and Poll Status

The build takes **60-90 minutes** (dominated by pulling the 16 GB NGC base image).
Run the poll cell every 5-10 minutes to check progress.

**Build stages:**
1. PRE_BUILD — ECR login + NGC login (reads `ngc-api-key` from Secrets Manager)
2. BUILD — `docker build` pulls `nvcr.io/nvidia/isaac-lab:2.1.0`, installs deps
3. POST_BUILD — `docker push` to ECR

**Common failures:**
- PRE_BUILD exit 254 → NGC secret missing or role lacks `secretsmanager:GetSecretValue` (Section 1a)
- BUILD fails on COPY → zip structure wrong (re-run Section 3)
- Push fails → ECR auth error (role lacks `ecr:*` permissions)

In [ ]:
build = cb.start_build(projectName=CB_PROJECT)
BUILD_ID = build['build']['id']
print(f' Build started: {BUILD_ID}')
print(f' Status: {build["build"]["buildStatus"]}')
print()
print(f'Monitor in console:')
print(f' https://{REGION}.console.aws.amazon.com/codesuite/codebuild/projects/{CB_PROJECT}/history?region={REGION}')
print()
print('Re-run the poll cell below every 5-10 min (NGC pull takes ~30-40 min).')


In [ ]:
# Re-run this cell to check build status
b = cb.batch_get_builds(ids=[BUILD_ID])['builds'][0]
status = b['buildStatus']
phase = b.get('currentPhase', '')
elapsed = int((time.time() - b['startTime'].timestamp()) / 60)

print(f'Status: {status}')
print(f'Phase: {phase}')
print(f'Elapsed: {elapsed} min')

if status == 'SUCCEEDED':
 imgs = ecr.describe_images(
 repositoryName=ECR_REPO,
 imageIds=[{'imageTag': 'latest'}]
 )['imageDetails']
 img = imgs[0]
 print(f'\n Build SUCCEEDED!')
 print(f' Image size: {img["imageSizeInBytes"] / 1e9:.1f} GB')
 print(f' Pushed at: {img["imagePushedAt"].strftime("%Y-%m-%d %H:%M UTC")}')
 print(f' ECR URI: {ECR_URI}:latest')
 print('\nYou can now run Lab4_RL_Training.ipynb')

elif status == 'FAILED':
 print('\n Build FAILED')
 for ph in b.get('phases', []):
 if ph.get('phaseStatus') == 'FAILED':
 print(f' Failed phase: {ph["phaseType"]}')
 for ctx in ph.get('contexts', []):
 print(f' Error: {ctx.get("message", "")}')
 print()
 print('See the README.md for common failure causes and fixes.')

else:
 print('\n(Re-run in 5 min — NGC pull takes ~30-40 min)')
